<a href="https://colab.research.google.com/github/rubenchov/Python-Remotesensing/blob/main/DownloadGEE_SARBajoCauca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
from google.colab import drive
import ee
import geemap
import pandas as pd
import numpy as np
import random
import cv2
from google.colab.patches import cv2_imshow
import os

In [ ]:
#IMPORTANT!
#1. USE COLAB WITH A GOOGLE PERSONAL ACCOUNT (NOT @elpoli.edu.co)
#2. SIGN IN TO https://developers.google.com/earth-engine
#3. CREATE A PROJECT, IN THIS CASE I NAMED IT "rubenchov". USE YOUR OWN.

# Drive
drive.mount('/content/drive')

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='rubenchov')

Mounted at /content/drive


# Function Buscar

In [ ]:
#Usar esta
def look_SAR_roi(start_date, end_date, lat1, lat2, long1, long2, pol = 'VH', dir = 'DESCENDING'):
  ROI = ee.Geometry.Rectangle(long1, lat2, long2, lat1)
  sar = (ee.ImageCollection('COPERNICUS/S1_GRD').
        filter(ee.Filter.listContains('transmitterReceiverPolarisation', pol)).
        filterBounds(ROI).filterDate(start_date, end_date).
        filter(ee.Filter.eq('instrumentMode', 'IW')).
        filter(ee.Filter.eq('orbitProperties_pass', dir)).
        #filter(ee.Filter.eq('resolution_meters', 10)).
        select(pol))
  print('Images found: ', sar.size().getInfo())
  if sar.size().getInfo() == 0:
    return None
  else:
    return sar.first().clip(ROI)

def BuscarSAR(inicio, fin, lat1, lon1, lat2, lon2, pol = 'VH', dir = 'DESCENDING'):
  p11 = ee.Geometry.Point(lon1, lat1)
  p12 = ee.Geometry.Point(lon1, lat2)
  p21 = ee.Geometry.Point(lon2, lat1)
  p22 = ee.Geometry.Point(lon2, lat2)

  sar = (ee.ImageCollection('COPERNICUS/S1_GRD').
        filter(ee.Filter.listContains('transmitterReceiverPolarisation', pol)).
        filterBounds(p11).filterBounds(p12).filterBounds(p21).filterBounds(p22).filterDate(inicio, fin).
        filter(ee.Filter.eq('instrumentMode', 'IW')).
        filter(ee.Filter.eq('orbitProperties_pass', dir)).
        select(pol))

  ROI = ee.Geometry.Rectangle(lon1, lat1, lon2, lat2)
  print('Images found: ', sar.size().getInfo())
  if sar.size().getInfo() == 0:
    return None
  else:
    return sar.first().clip(ROI)

# Download SAR

In [ ]:
#Región dentro del Bajo Cauca Antioqueño
#zone = 'Confluencia'
#zone = 'Minas'
zone = 'Bagre'

folder = zone + '/'
dirname = os.path.join(os.getcwd(), '/content/drive/MyDrive/2026/Docencia/Visión con IA/6. Teledetección con GEE/Ejemplo Análisis Bajo Cauca/' + folder) #Update to your own working directory

if zone == 'Confluencia':
  #Confluencia
  lat1, long1, lat2, long2 = 8.113206, -74.851446, 8.037527, -74.746023
elif zone == 'Minas':
  #Minas
  lat1, long1, lat2, long2 = 7.993460, -74.890861, 7.846429, -74.781447
elif zone == 'Bagre':
  #Bagre
  lat1, long1, lat2, long2 = 7.622440, -74.865486, 7.553838, -74.790542

#Export only one image

In [ ]:
#Una sola (OPCIONAL!)
year = '2025'
month = '07'
first_day = '10'
last_day = '25'

start_date = year + '-' + month + '-' + first_day
end_date = year + '-' + month + '-' + last_day

sarVH = BuscarSAR(start_date, end_date, lat1, long1, lat2, long2, pol = 'VH', dir = 'ASCENDING') #Jean
#sarVH = look_SAR_roi(start_date, end_date, lat1, lat2, long1, long2, pol = 'VH', dir = 'ASCENDING') #Rubén
sarVH_vis = sarVH.visualize(**{'min' : -25, 'max' : 5})
info = sarVH_vis.getInfo()
print(info['bands'][0]['dimensions'])

#Export image
export_path = dirname + year + '-' + month + '.tif'
geemap.ee_export_image(sarVH_vis, export_path, scale = 10, file_per_band=False)

Images found:  2
[828, 760]
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/2026/Docencia/Visión e IA/6. Teledetección con GEE/Ejemplo Análisis Bajo Cauca/Bagre/2025-07.tif


#Export several images. One per month various years

In [ ]:
years = ['2024', '2025', '2026'] #Algunos
#years = ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026'] #Todos
months = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']

for year in years:
  print('Year: ', year)
  for month in months:
    print('Month: ', month)

    first_day = '01'
    if month == '02':
      last_day = '28'
    else:
      last_day = '30'

    start_date = year + '-' + month + '-' + first_day
    end_date = year + '-' + month + '-' + last_day

    # VH
    sarVH = look_SAR_roi(start_date, end_date, lat1, lat2, long1, long2, pol = 'VH', dir = 'ASCENDING') #Rubén
    #sarVH = BuscarSAR(start_date, end_date, lat1, long1, lat2, long2, pol = 'VH', dir = 'ASCENDING') #Jean
    if sarVH is not None:
      sarVH_vis = sarVH.visualize(**{'min' : -25, 'max' : 5})
      info = sarVH_vis.getInfo()
      print(info['bands'][0]['dimensions']) #Buscar tamaño con función Rubén
      # Export
      export_path = filename = dirname + year + '-' + month + '.tif'
      geemap.ee_export_image(sarVH_vis, export_path, scale = 10, file_per_band=False)

Year:  2024
Month:  01
Images found:  2
[828, 760]
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/2026/Docencia/Visión con IA/6. Teledetección con GEE/Ejemplo Análisis Bajo Cauca/Bagre/2024-01.tif
Month:  02
Images found:  3
[828, 760]
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/2026/Docencia/Visión con IA/6. Teledetección con GEE/Ejemplo Análisis Bajo Cauca/Bagre/2024-02.tif
Month:  03
Images found:  2
[828, 760]
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/2026/Docencia/Visión con IA/6. Teledetección con GEE/Ejemplo Análisis Bajo Cauca/Bagre/2024-03.tif
Month:  04
Images found:  3
[828, 760]
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/2026/Docencia/Visión con IA/6. Teledetección con GEE/Ejemplo Análisis Bajo Cauca/Bagre/2024-04.tif
Month:  05
Images found:  2
[828, 760]
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/2026/Docenci